# PoC B 模型测试

可从已发布 Model Repository 或本地 checkpoint 加载。加载器会读取固定 Qwen revision，并始终返回封闭集 Top-1 AppID。

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

HF_REPO_ID = 'hxgdzyuyi/qwen3-8b-steam-entity-linking-poc-b'
LOCAL_CHECKPOINT: Path | None = None
TOP_K = 5
TEST_TEXTS = ['Counter-Strike 2', 'CS2', '反恐精英', '那个拆包的射击游戏']
RUN_OFFICIAL_EVALUATION = False
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/qwen-steam-entity-linking')]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'poc_b/scripts/steam_entity_classifier.py').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('未找到项目。')
POC_DIR = PROJECT_DIR / 'poc_b'
os.environ.setdefault('HF_HOME', '/workspace/.cache/huggingface')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(POC_DIR / 'requirements-cloud.txt')], cwd=PROJECT_DIR, check=True)

In [ ]:
sys.path.insert(0, str(POC_DIR / 'scripts'))
from steam_entity_classifier import SteamEntityLinker

source = LOCAL_CHECKPOINT.resolve() if LOCAL_CHECKPOINT else HF_REPO_ID
linker = SteamEntityLinker.from_pretrained(source)
predictions = linker.predict(TEST_TEXTS, top_k=TOP_K)
print(json.dumps(predictions, ensure_ascii=False, indent=2))

In [ ]:
if RUN_OFFICIAL_EVALUATION:
    if LOCAL_CHECKPOINT is None:
        raise RuntimeError('官方评测需要把 LOCAL_CHECKPOINT 指向 run/checkpoints/epoch-N。')
    run_dir = LOCAL_CHECKPOINT.resolve().parents[1]
    subprocess.run([sys.executable, str(POC_DIR / 'scripts/evaluate.py'), '--run-dir', str(run_dir), '--checkpoint', str(LOCAL_CHECKPOINT)], cwd=PROJECT_DIR, check=True)
else:
    print('RUN_OFFICIAL_EVALUATION=False：仅执行交互预测。')